# STIC validation on Ameriflux sites

Validation of STIC using field data from the ameriflux database and landsat acquisitions

In [ ]:
# import os
# os.environ["NUMBA_DISABLE_JIT"] = "1"
# os.environ["NUMBA_DEVELOPER_MODE"] = "1"
# os.environ["NUMBA_DEBUG"] = "1"

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import warnings

import ipywidgets as widgets
import numpy as np
import numpy.typing as npt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import clear_output, display
from plotly.subplots import make_subplots
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sweat.common import utils
from sweat.common.flux import CST_SB, compute_et_from_le
from sweat.stic.constant import KELVIN_CST
from sweat.stic.runner import run_batch_init_model, run_batch_model

## Functions

In [ ]:
# Prepare function


def prepare_input_data(df: pd.DataFrame, dataset: str = "fl") -> pd.DataFrame:
    """
    Prepare data to run STIC
    """
    # Get reflectances, lst, fc and lai
    output_df = df[
        [
            "name",
            "lat",
            "lon",
            "datetime_utc",
            "local_time",
            "datetime",
            "elevation",
            "landcover",
            "blue",
            "green",
            "red",
            "nir",
            "swir",
            "ts",
            "fc",
            "lai",
        ]
    ].copy()
    # Get EC tower data
    if dataset == "am":
        output_df = pd.concat(
            [
                output_df,
                df[
                    [
                        "am_le",
                        "am_h",
                        "am_g",
                        "am_rn",
                        "am_srn",
                        "am_sw_out",
                        "am_sw_in",
                        "am_lw_out",
                        "am_lw_in",
                        "am_ta",
                        "am_ts",
                        "am_td",
                        "am_rh",
                        "am_et",
                        "am_rn-g",
                        "am_le_closed",
                    ]
                ],
            ],
            axis=1,
        )
        output_df = output_df.rename(
            columns={
                "am_le": "ec_le",
                "am_h": "ec_h",
                "am_g": "ec_g",
                "am_rn": "rn",
                "am_srn": "srn",
                "am_sw_out": "sw_out",
                "am_sw_in": "sw_in",
                "am_lw_out": "lw_out",
                "am_lw_in": "lw_in",
                "am_ta": "ta",
                "am_td": "td",
                "am_rh": "rh",
                "am_rn-g": "ec_rn-g",
                "am_le_closed": "ec_le_closed",
            }
        )
    elif dataset == "fl":
        output_df = pd.concat(
            [
                output_df,
                df[
                    [
                        "fl_le",
                        "fl_h",
                        "fl_g",
                        "fl_rn",
                        "fl_srn",
                        "fl_sw_out",
                        "fl_sw_in",
                        "fl_lw_out",
                        "fl_lw_in",
                        "fl_ta",
                        "fl_td",
                        "fl_rh",
                        "fl_le_closed",
                        "fl_rn-g",
                    ]
                ],
            ],
            axis=1,
        )
        output_df = output_df.rename(
            columns={
                "fl_le": "ec_le",
                "fl_h": "ec_h",
                "fl_g": "ec_g",
                "fl_rn": "rn",
                "fl_srn": "srn",
                "fl_sw_out": "sw_out",
                "fl_sw_in": "sw_in",
                "fl_lw_out": "lw_out",
                "fl_lw_in": "lw_in",
                "fl_ta": "ta",
                "fl_td": "td",
                "fl_rh": "rh",
                "fl_rn-g": "ec_rn-g",
                "fl_le_closed": "ec_le_closed",
            }
        )
    else:
        msg = "Dataset unkown"
        raise ValueError(msg)
    # ln
    output_df["ln"] = output_df["lw_in"] - output_df["lw_in"]
    # emissivity
    output_df["emis"] = output_df["lw_out"] / (
        CST_SB * (output_df["ts"] + KELVIN_CST) ** 4
    )
    # Vegetation indices
    output_df["vari_green"] = utils.compute_vari_green(
        red=output_df["red"], green=output_df["green"], blue=output_df["blue"]
    )
    output_df["gli"] = utils.compute_gli(
        red=output_df["red"], green=output_df["green"], blue=output_df["blue"]
    )
    output_df["ndvi"] = utils.compute_ndvi(
        red=output_df["red"], nir=output_df["nir"]
    )
    output_df["gndvi"] = utils.compute_gndvi(
        nir=output_df["nir"], green=output_df["green"]
    )
    output_df["msavi"] = utils.compute_msavi(
        red=output_df["red"], nir=output_df["green"]
    )
    # Rename columns
    output_df = output_df.rename(
        columns={"datetime": "date", "datetime_utc": "date (utc)"}
    )
    # Remove nan inputs
    output_df = output_df.dropna(
        subset=[
            "ts",
            "ta",
            "td",
            "rh",
            "fc",
            "lai",
            "srn",
            "rn",
            "ln",
            "local_time",
            "blue",
            "green",
            "red",
            "nir",
            "swir",
            "vari_green",
            "gli",
            "ndvi",
            "gndvi",
            "msavi",
            "emis",
        ],
        how="any",
    )
    # reorder columns
    return output_df[
        [
            "name",
            "date",
            "date (utc)",
            "lat",
            "lon",
            "landcover",
            "ts",
            "ta",
            "td",
            "rh",
            "fc",
            "lai",
            "srn",
            "rn",
            "ln",
            "local_time",
            "blue",
            "green",
            "red",
            "nir",
            "swir",
            "vari_green",
            "gli",
            "ndvi",
            "gndvi",
            "msavi",
            "emis",
            "ec_le",
            "ec_h",
            "ec_le_closed",
            "ec_g",
            "ec_rn-g",
        ]
    ]


In [ ]:
# Plot functions


def safe_r2_score(y_true, y_pred):
    idx = np.isfinite(y_true) & np.isfinite(y_pred)

    if np.sum(idx) < 2:
        return np.nan

    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always", UndefinedMetricWarning)

        r2 = r2_score(y_true[idx], y_pred[idx])

        if any(issubclass(warn.category, UndefinedMetricWarning) for warn in w):
            return np.nan

    return r2


def safe_polyfit(x, y):
    idx = np.isfinite(x) & np.isfinite(y)

    if np.sum(idx) < 2:
        return np.nan, np.nan

    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always", np.exceptions.RankWarning)

        slope, intercept = np.polyfit(x[idx], y[idx], 1)

        if any(issubclass(warn.category, np.RankWarning) for warn in w):
            return np.nan, np.nan

    return slope, intercept


def compute_metrics(
    measured: npt.ArrayLike, estimated: npt.ArrayLike
) -> tuple[float, float, float, float, float]:
    """
    Compute slope, mbe, mae, rmse, r2
    """
    slope, _ = safe_polyfit(measured, estimated)
    if len(measured - estimated) > 0 and not np.all(
        np.isnan(measured - estimated)
    ):
        mbe = np.nanmean(estimated - measured)
    else:
        mbe = np.nan
    idx = np.isfinite(measured) & np.isfinite(estimated)
    if len(measured[idx]) > 0 and len(estimated[idx]):
        mae = mean_absolute_error(measured[idx], estimated[idx])
        rmse = np.sqrt(mean_squared_error(measured[idx], estimated[idx]))
        r2 = safe_r2_score(measured[idx], estimated[idx])
    else:
        mae = np.nan
        rmse = np.nan
        r2 = np.nan
    return (slope, mbe, mae, rmse, r2)


def generate_metrics_table(
    df: pd.DataFrame, variables: list[str] | None = None
) -> pd.DataFrame:
    """
    Generate metrics table
    """
    if variables is None:
        variables = ["le", "h", "le_closed"]
    metrics = []
    # Compute by site
    for name, group in df.groupby(["name", "landcover"]):
        for var in variables:
            est_var = var if "_closed" not in var else var[:-7]
            slope, mbe, mae, rmse, r2 = compute_metrics(
                measured=group[f"ec_{var}"], estimated=group[est_var]
            )
            metrics.append(
                {
                    "name": name[0],
                    "landcover": name[1],
                    "variable": var,
                    "slope": slope,
                    "mbe": mbe,
                    "mae": mae,
                    "rmse": rmse,
                    "r2": r2,
                }
            )
    # Compute for all
    for var in variables:
        est_var = var if "_closed" not in var else var[:-7]
        slope, mbe, mae, rmse, r2 = compute_metrics(
            measured=group[f"ec_{var}"], estimated=group[est_var]
        )
        metrics.append(
            {
                "name": "all",
                "landcover": "all",
                "variable": var,
                "slope": slope,
                "mbe": mbe,
                "mae": mae,
                "rmse": rmse,
                "r2": r2,
            }
        )
    df_metrics = pd.DataFrame.from_records(metrics)
    df_metrics.attrs["label"] = df.attrs["label"]
    return df_metrics


def plot_data(df: pd.DataFrame, variable: str | None = None):
    """
    Plot data
    """
    sites = sorted(df["name"].unique())
    variables = sorted(
        df.columns.difference(
            ["name", "date", "data (utc)", "lat", "lon", "landcover"]
        )
    )
    if variable is None or variable not in variables:
        variable = "ts"
    site_dropdown = widgets.Dropdown(
        options=sites,
        value=sites[0],
        description="Site:",
        style={"description_width": "initial"},
    )

    var_dropdown = widgets.Dropdown(
        options=variables,
        value=variable,
        description="Variable:",
        style={"description_width": "initial"},
    )

    # Output area for plot
    output = widgets.Output()

    # --- Update function ---
    def update_plot(change=None):  # noqa
        with output:
            clear_output(wait=True)

            site = site_dropdown.value
            var = var_dropdown.value

            subset = df[df["name"] == site]

            fig = px.line(
                subset,
                x="date",
                y=var,
                markers=True,
            )

            # Print value
            fig.update_traces(
                hovertemplate=(
                    f"<b>Site:</b> {site}<br>"
                    "<b>Date:</b> %{x|%Y-%m-%d}<br>"
                    f"<b>{var}:</b> %{{y:.2f}}<br>"
                    "<extra></extra>"
                )
            )

            # Format x-axis as YYYY-MM-DD
            fig.update_xaxes(tickformat="%Y-%m-%d")

            # Improve layout
            fig.update_layout(
                autosize=True,
                margin={"l": 20, "r": 20, "t": 60, "b": 40},
                xaxis_title="Date",
                yaxis_title=var,
                title={
                    "text": f"Evolution of {var} over time (site: {site})",
                    "x": 0.5,
                    "xanchor": "center",
                    "font": {"size": 20},
                },
            )
            fig.show(config={"responsive": True})

    # --- Link widgets to function ---
    site_dropdown.observe(update_plot, names="value")
    var_dropdown.observe(update_plot, names="value")

    # --- Layout ---
    controls = widgets.HBox([site_dropdown, var_dropdown])

    display(controls, output)

    # Initial plot
    update_plot()


def plot_data_by_class(df: pd.DataFrame, variable: str | None = None):
    """
    Plot data
    """
    classes = ["All", *sorted(df["landcover"].unique())]
    variables = sorted(
        df.columns.difference(
            ["name", "date", "data (utc)", "lat", "lon", "landcover"]
        )
    )
    if variable is None or variable not in variables:
        variable = "ts"

    class_dropdown = widgets.Dropdown(
        options=classes,
        value="All",
        description="Class:",
        style={"description_width": "initial"},
    )

    var_dropdown = widgets.Dropdown(
        options=variables,
        value="ts",
        description="Variable:",
        style={"description_width": "initial"},
    )

    # Output area for plot
    output = widgets.Output()

    # --- Update function ---
    def update_plot(change=None):  # noqa
        with output:
            clear_output(wait=True)

            selected_class = class_dropdown.value
            var = var_dropdown.value

            if selected_class == "All":
                subset = df
            else:
                subset = df[df["landcover"] == selected_class]

            fig = px.line(
                subset,
                x="date",
                y=var,
                markers=True,
                color="name",
                render_mode="webgl",
            )

            # Print value
            fig.update_traces(
                hovertemplate=(
                    "<b>Site:</b> %{fullData.name}<br>"
                    "<b>Date:</b> %{x|%Y-%m-%d}<br>"
                    f"<b>{var}:</b> %{{y:.2f}}<br>"
                    "<extra></extra>"
                )
            )

            # Format x-axis as YYYY-MM-DD
            fig.update_xaxes(tickformat="%Y-%m-%d")

            # Improve layout
            fig.update_layout(
                autosize=True,
                margin={"l": 20, "r": 20, "t": 60, "b": 40},
                xaxis_title="Date",
                yaxis_title=var,
                legend_title="Site",
                # legend=dict(orientation="h"),
                title={
                    "text": f"Evolution of {var} over time (Class: {selected_class})",
                    "x": 0.5,
                    "xanchor": "center",
                    "font": {"size": 20},
                },
            )
            fig.show(config={"responsive": True})

    # --- Link widgets to function ---
    class_dropdown.observe(update_plot, names="value")
    var_dropdown.observe(update_plot, names="value")

    # --- Layout ---
    controls = widgets.HBox([class_dropdown, var_dropdown])

    display(controls, output)

    # Initial plot
    update_plot()


def plot_scatter(
    df: pd.DataFrame | list[pd.DataFrame],
    x: str | None = None,
    y: str | None = None,
):
    """
    Scatter plot
    """
    # --- Normalize input ---
    if isinstance(df, pd.DataFrame):
        dfs = [df]
    elif isinstance(df, (list, tuple)) and len(df) in [1, 2]:
        dfs = list(df)
    else:
        msg = "df must be a DataFrame or a list/tuple of 1 or 2 DataFrames"
        raise ValueError(msg)
    names = [dfi.attrs.get("label", f"DF{i + 1}") for i, dfi in enumerate(dfs)]

    # --- Widgets (based on first df) ---
    ref_df = dfs[0]

    sites = sorted(ref_df["name"].unique())
    variables = sorted(
        ref_df.columns.difference(
            ["name", "date", "data (utc)", "lat", "lon", "landcover"]
        )
    )

    if x is None or x not in variables:
        x = "ta"
    if y is None or y not in variables:
        y = "ts"

    site_dropdown = widgets.Dropdown(
        options=sites,
        value=sites[0],
        description="Site:",
        style={"description_width": "initial"},
    )

    xvar_dropdown = widgets.Dropdown(
        options=variables,
        value=x,
        description="X axis:",
        style={"description_width": "initial"},
    )

    yvar_dropdown = widgets.Dropdown(
        options=variables,
        value=y,
        description="Y axis:",
        style={"description_width": "initial"},
    )

    metrics_checkbox = widgets.Checkbox(
        value=False, description="Compute metrics"
    )

    range_checkbox = widgets.Checkbox(value=False, description="Fixed range")

    output = widgets.Output()

    # --- Update function ---
    def update_plot(change=None):  # noqa
        with output:
            clear_output(wait=True)

            site = site_dropdown.value
            xvar = xvar_dropdown.value
            yvar = yvar_dropdown.value

            # --- Create figure ---
            if len(dfs) == 2:
                fig = make_subplots(rows=1, cols=2, subplot_titles=names)
            else:
                fig = go.Figure()

            # --- Loop over dataframes ---
            for i, dfi in enumerate(dfs):
                subset = dfi[dfi["name"] == site]
                x_values = subset[xvar]
                y_values = subset[yvar]

                row, col = (1, i + 1) if len(dfs) == 2 else (None, None)

                trace = go.Scatter(
                    x=x_values,
                    y=y_values,
                    mode="markers",
                    marker={"size": 6, "opacity": 0.5},
                    customdata=subset["date"],
                    showlegend=False,
                )

                if len(dfs) == 2:
                    fig.add_trace(trace, row=row, col=col)
                else:
                    fig.add_trace(trace)

                # Hover
                fig.update_traces(
                    hovertemplate=(
                        f"<b>Site:</b> {site}<br>"
                        f"<b>{xvar}:</b> %{{x:.2f}}<br>"
                        f"<b>{yvar}:</b> %{{y:.2f}}<br>"
                        "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                    )
                )

                # Fixed range
                if range_checkbox.value:
                    lims = [
                        min(x_values.min(), y_values.min()),
                        max(x_values.max(), y_values.max()),
                    ]
                    axis_range = [
                        lims[0] - 0.1 * abs(lims[0]),
                        lims[1] + 0.1 * abs(lims[1]),
                    ]

                    if len(dfs) == 2:
                        fig.update_xaxes(range=axis_range, row=row, col=col)
                        fig.update_yaxes(range=axis_range, row=row, col=col)
                    else:
                        fig.update_xaxes(range=axis_range)
                        fig.update_yaxes(range=axis_range)

                # Metrics
                if metrics_checkbox.value:
                    slope, intercept = safe_polyfit(x_values, y_values)

                    lims = [
                        min(x_values.min(), y_values.min()),
                        max(x_values.max(), y_values.max()),
                    ]

                    # Identity line
                    line1 = go.Scatter(
                        x=lims,
                        y=lims,
                        mode="lines",
                        line={"dash": "dash", "color": "black"},
                        showlegend=False,
                    )

                    # Fit line
                    line2 = None
                    if not np.isnan(slope) or np.isnan(intercept):
                        line2 = go.Scatter(
                            x=lims,
                            y=[
                                lims[0] * slope + intercept,
                                lims[1] * slope + intercept,
                            ],
                            mode="lines",
                            line={"color": "red"},
                            showlegend=False,
                        )

                    if len(dfs) == 2:
                        fig.add_trace(line1, row=row, col=col)
                        if line2:
                            fig.add_trace(line2, row=row, col=col)
                    else:
                        fig.add_trace(line1)
                        if line2:
                            fig.add_trace(line2)

                    # --- Metrics text ---
                    slope_m, mbe, mae, rmse, r2 = compute_metrics(
                        x_values, y_values
                    )

                    if len(dfs) == 2:
                        fig.add_annotation(
                            x=0.97,
                            y=0.03,
                            xref="x domain",
                            yref="y domain",
                            text=(
                                f"slope = {slope_m:.2f}<br>"
                                f"MBE = {mbe:.2f}<br>"
                                f"MAE = {mae:.2f}<br>"
                                f"RMSE = {rmse:.2f}<br>"
                                f"R² = {r2:.2f}"
                            ),
                            showarrow=False,
                            align="right",
                            row=row,
                            col=col,
                        )
                    else:
                        fig.add_annotation(
                            x=0.97,
                            y=0.03,
                            xref="x domain",
                            yref="y domain",
                            text=(
                                f"slope = {slope_m:.2f}<br>"
                                f"MBE = {mbe:.2f}<br>"
                                f"MAE = {mae:.2f}<br>"
                                f"RMSE = {rmse:.2f}<br>"
                                f"R² = {r2:.2f}"
                            ),
                            showarrow=False,
                            align="right",
                        )
            # Layout
            fig.update_layout(
                template="plotly_white",
                height=700,
                width=1400 if len(dfs) == 2 else 700,
            )
            if len(dfs) == 2:
                fig.update_xaxes(title_text=xvar, row=row, col=col)
                fig.update_yaxes(title_text=yvar, row=row, col=col)
            else:
                fig.update_xaxes(title_text=xvar)
                fig.update_yaxes(title_text=yvar)

            fig.show(config={"responsive": True})

    # --- Bind widgets ---
    for w in [
        site_dropdown,
        xvar_dropdown,
        yvar_dropdown,
        metrics_checkbox,
        range_checkbox,
    ]:
        w.observe(update_plot, names="value")

    controls = widgets.HBox(
        [
            site_dropdown,
            xvar_dropdown,
            yvar_dropdown,
            metrics_checkbox,
            range_checkbox,
        ]
    )

    display(controls, output)

    # Initial plot
    update_plot()


def plot_scatter_by_class(
    df: pd.DataFrame | list[pd.DataFrame],
    x: str | None = None,
    y: str | None = None,
):
    """
    Scatter plot by class
    """
    # --- Normalize input ---
    if isinstance(df, pd.DataFrame):
        dfs = [df]
    elif isinstance(df, (list, tuple)) and len(df) in [1, 2]:
        dfs = list(df)
    else:
        msg = "df must be a DataFrame or a list/tuple of 1 or 2 DataFrames"
        raise ValueError(msg)
    names = [
        dfi.attrs.get("label", f"Data {i + 1}") for i, dfi in enumerate(dfs)
    ]

    # --- Widgets (based on first df) ---
    ref_df = dfs[0]

    classes = ["All", *sorted(ref_df["landcover"].unique())]
    variables = sorted(
        ref_df.columns.difference(
            ["name", "date", "data (utc)", "lat", "lon", "landcover"]
        )
    )

    if x is None or x not in variables:
        x = "ta"
    if y is None or y not in variables:
        y = "ts"

    class_dropdown = widgets.Dropdown(
        options=classes,
        value=classes[0],
        description="Site:",
        style={"description_width": "initial"},
    )

    xvar_dropdown = widgets.Dropdown(
        options=variables,
        value=x,
        description="X axis:",
        style={"description_width": "initial"},
    )

    yvar_dropdown = widgets.Dropdown(
        options=variables,
        value=y,
        description="Y axis:",
        style={"description_width": "initial"},
    )

    metrics_checkbox = widgets.Checkbox(
        value=False, description="Compute metrics"
    )

    range_checkbox = widgets.Checkbox(value=False, description="Fixed range")

    output = widgets.Output()

    # --- Update function ---
    def update_plot(change=None):  # noqa
        with output:
            clear_output(wait=True)

            selected_class = class_dropdown.value
            xvar = xvar_dropdown.value
            yvar = yvar_dropdown.value

            # --- Create figure ---
            if len(dfs) == 2:
                fig = make_subplots(rows=1, cols=2, subplot_titles=names)
            else:
                fig = go.Figure()

            # --- Loop over dataframes ---
            for i, dfi in enumerate(dfs):
                if selected_class == "All":
                    subset = dfi
                else:
                    subset = dfi[dfi["landcover"] == selected_class]
                x_values = subset[xvar]
                y_values = subset[yvar]

                row, col = (1, i + 1) if len(dfs) == 2 else (None, None)

                for site_name, group in subset.groupby("name"):
                    trace = go.Scatter(
                        x=group[xvar],
                        y=group[yvar],
                        mode="markers",
                        name=site_name,
                        marker={"size": 6, "opacity": 0.5},
                        customdata=group["date"],
                    )

                    if len(dfs) == 2:
                        fig.add_trace(trace, row=row, col=col)
                    else:
                        fig.add_trace(trace)

                # Hover
                fig.update_traces(
                    hovertemplate=(
                        "<b>Site:</b> %{fullData.name}<br>"
                        f"<b>{xvar}:</b> %{{x:.2f}}<br>"
                        f"<b>{yvar}:</b> %{{y:.2f}}<br>"
                        "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                    )
                )

                # Fixed range
                if range_checkbox.value:
                    lims = [
                        min(x_values.min(), y_values.min()),
                        max(x_values.max(), y_values.max()),
                    ]
                    axis_range = [
                        lims[0] - 0.1 * abs(lims[0]),
                        lims[1] + 0.1 * abs(lims[1]),
                    ]

                    if len(dfs) == 2:
                        fig.update_xaxes(range=axis_range, row=row, col=col)
                        fig.update_yaxes(range=axis_range, row=row, col=col)
                    else:
                        fig.update_xaxes(range=axis_range)
                        fig.update_yaxes(range=axis_range)

                # Metrics
                if metrics_checkbox.value:
                    slope, intercept = safe_polyfit(x_values, y_values)

                    lims = [
                        min(x_values.min(), y_values.min()),
                        max(x_values.max(), y_values.max()),
                    ]

                    # Identity line
                    line1 = go.Scatter(
                        x=lims,
                        y=lims,
                        mode="lines",
                        line={"dash": "dash", "color": "black"},
                        showlegend=False,
                    )

                    # Fit line
                    line2 = None
                    if np.isnan(slope) or np.isnan(intercept):
                        line2 = go.Scatter(
                            x=lims,
                            y=[
                                lims[0] * slope + intercept,
                                lims[1] * slope + intercept,
                            ],
                            mode="lines",
                            line={"color": "red"},
                            showlegend=False,
                        )

                    if len(dfs) == 2:
                        fig.add_trace(line1, row=row, col=col)
                        if line2:
                            fig.add_trace(line2, row=row, col=col)
                    else:
                        fig.add_trace(line1)
                        if line2:
                            fig.add_trace(line2)

                    # --- Metrics text ---
                    slope_m, mbe, mae, rmse, r2 = compute_metrics(
                        x_values, y_values
                    )

                    if len(dfs) == 2:
                        fig.add_annotation(
                            x=0.97,
                            y=0.03,
                            xref="x domain",
                            yref="y domain",
                            text=(
                                f"slope = {slope_m:.2f}<br>"
                                f"MBE = {mbe:.2f}<br>"
                                f"MAE = {mae:.2f}<br>"
                                f"RMSE = {rmse:.2f}<br>"
                                f"R² = {r2:.2f}"
                            ),
                            showarrow=False,
                            align="right",
                            row=row,
                            col=col,
                        )
                    else:
                        fig.add_annotation(
                            x=0.97,
                            y=0.03,
                            xref="x domain",
                            yref="y domain",
                            text=(
                                f"slope = {slope_m:.2f}<br>"
                                f"MBE = {mbe:.2f}<br>"
                                f"MAE = {mae:.2f}<br>"
                                f"RMSE = {rmse:.2f}<br>"
                                f"R² = {r2:.2f}"
                            ),
                            showarrow=False,
                            align="right",
                        )
            # Layout
            fig.update_layout(
                template="plotly_white",
                height=700,
                width=1400 if len(dfs) == 2 else 700,
            )
            if len(dfs) == 2:
                fig.update_xaxes(title_text=xvar, row=row, col=col)
                fig.update_yaxes(title_text=yvar, row=row, col=col)
            else:
                fig.update_xaxes(title_text=xvar)
                fig.update_yaxes(title_text=yvar)

            fig.show(config={"responsive": True})

    # --- Bind widgets ---
    for w in [
        class_dropdown,
        xvar_dropdown,
        yvar_dropdown,
        metrics_checkbox,
        range_checkbox,
    ]:
        w.observe(update_plot, names="value")

    controls = widgets.HBox(
        [
            class_dropdown,
            xvar_dropdown,
            yvar_dropdown,
            metrics_checkbox,
            range_checkbox,
        ]
    )

    display(controls, output)

    # Initial plot
    update_plot()


def init_stic(df: pd.DataFrame, version: str) -> pd.DataFrame:
    """
    Compute STIC model
    """
    res = run_batch_init_model(data=df, version=version)
    output_df = pd.DataFrame(
        {
            "name": df["name"].values,
            "date": df["date"],
            "le": res[:, 0],
            "h": res[:, 1],
            "g": res[:, 2],
            "rn": df["rn"].values,
            "ta": df["ta"].values,
            "td": df["td"].values,
            "rh": df["rh"].values,
            "ts": df["ts"].values,
            "ga": res[:, 3],
            "gs": res[:, 4],
            "t0": res[:, 5],
            "t0d": res[:, 6],
            "m": res[:, 7],
            "m_canopy": res[:, 8],
            "m_soil": res[:, 9],
            "m_surf": res[:, 10],
            "m_rz": res[:, 11],
            "da": res[:, 12],
            "ds": res[:, 13],
            "es": res[:, 14],
            "ea": res[:, 15],
            "e0": res[:, 16],
            "esstar": res[:, 17],
            "e0star": res[:, 18],
            "alpha": res[:, 19],
        }
    )
    cols = list(df.columns.difference(output_df.columns))
    cols.append("date")
    cols.append("name")
    output_df = pd.merge(left=output_df, right=df[cols], on=["date", "name"])
    output_df.attrs["label"] = f"STIC {version}"
    return output_df


def run_stic(df: pd.DataFrame, version: str) -> pd.DataFrame:
    """
    Compute STIC model
    """
    res = run_batch_model(data=df, threshold=0.05, nb_steps=15, version=version)
    output_df = pd.DataFrame(
        {
            "name": df["name"].values,
            "date": df["date"],
            "le": res[:, 0],
            "h": res[:, 1],
            "ef": res[:, 2],
            "g": res[:, 3],
            "rn": df["rn"].values,
            "ta": df["ta"].values,
            "td": df["td"].values,
            "rh": df["rh"].values,
            "ts": df["ts"].values,
            "ga": res[:, 4],
            "gs": res[:, 5],
            "t0": res[:, 6],
            "m": res[:, 7],
        }
    )
    output_df["et"] = output_df.apply(
        lambda x: compute_et_from_le(x["le"]), axis=1
    )
    output_df["rn-g"] = output_df["rn"] - output_df["g"]
    output_df["diff_le"] = output_df["le"] - df["ec_le"]
    output_df["diff_le_closed"] = output_df["le"] - df["ec_le_closed"]
    cols = list(df.columns.difference(output_df.columns))
    cols.append("date")
    cols.append("name")
    output_df = pd.merge(left=output_df, right=df[cols], on=["date", "name"])
    output_df.attrs["label"] = f"STIC {version}"
    return output_df


def plot_metrics(
    df: pd.DataFrame | list[pd.DataFrame], variable: str, metric: str
):
    """
    Plot metrics per landcover class
    """
    # --- Normalize input ---
    if isinstance(df, pd.DataFrame):
        dfs = [df]
    elif isinstance(df, (list, tuple)) and len(df) in [1, 2]:
        dfs = list(df)
    else:
        msg = "df must be a DataFrame or a list/tuple of 1 or 2 DataFrames"
        raise ValueError(msg)
    names = [dfi.attrs.get("label", f"DF{i + 1}") for i, dfi in enumerate(dfs)]

    df_mean = []
    for name, dfi in zip(names, dfs, strict=True):
        dfi_mean = (
            dfi[dfi["variable"] == variable]
            .groupby("landcover", as_index=False)[metric]
            .mean()
        )
        dfi_mean["source"] = name
        df_mean.append(dfi_mean)
    df_mean = pd.concat(df_mean, ignore_index=True)
    fig = px.bar(
        df_mean,
        x="landcover",
        y="rmse",
        color="source",
        barmode="group",
        title=f"Average RMSE for {variable.upper()} per landcover",
        labels={"rmse": "Mean RMSE", "landcover": "Landcover"},
    )

    fig.show()

## Input

In [ ]:
data_path = os.path.join(
    os.path.expanduser("~"), "data", "validation", "data_ameriflux.csv"
)

## Read data

In [ ]:
input_data = pd.read_csv(data_path, parse_dates=["datetime", "datetime_utc"])

In [ ]:
input_data

## Prepare data

In [ ]:
fl_data = prepare_input_data(input_data, dataset="fl")

In [ ]:
fl_data

In [ ]:
am_data = prepare_input_data(input_data, dataset="am")

In [ ]:
am_data

### Plot data

In [ ]:
plot_data(fl_data)

In [ ]:
plot_data_by_class(fl_data)

In [ ]:
plot_scatter(fl_data)

In [ ]:
plot_scatter_by_class(fl_data)

## Compute initialization of STIC model

In [ ]:
%%time
init_v1_3 = init_stic(fl_data, version="1.3")

In [ ]:
init_v1_3.head()

In [ ]:
%%time
init_v1_4 = init_stic(fl_data, version="1.4")

In [ ]:
init_v1_4.head()

In [ ]:
plot_scatter([init_v1_3, init_v1_4], x="ec_le", y="le")

In [ ]:
plot_scatter_by_class([init_v1_3, init_v1_4], x="ec_le", y="le")

## Compute STIC model

In [ ]:
%%time
res_v1_3 = run_stic(fl_data, version="1.3")

In [ ]:
res_v1_3

In [ ]:
%%time
res_v1_4 = run_stic(fl_data, version="1.4")

In [ ]:
res_v1_4

In [ ]:
plot_scatter([res_v1_3, res_v1_4], x="ec_le", y="le")

In [ ]:
plot_scatter_by_class([res_v1_3, res_v1_4], x="ec_le", y="le")

## Compute metrics

In [ ]:
metrics_v1_3 = generate_metrics_table(res_v1_3)
metrics_v1_4 = generate_metrics_table(res_v1_4)

In [ ]:
metrics_v1_3

In [ ]:
metrics_v1_4

In [ ]:
plot_metrics([metrics_v1_3, metrics_v1_4], "le", "rmse")

In [ ]:
plot_metrics([metrics_v1_3, metrics_v1_4], "le_closed", "rmse")